# Git Branching, Merging and Conflict Resolution

Implementation notebook for the existing virtual laboratory. The provided Django/Python topics are an implementation checklist, not additional experiments. Run with the repository virtual environment. All data below is temporary.

In [ ]:
from pathlib import Path
import os, sys, tempfile
ROOT=Path.cwd()
if not (ROOT/"manage.py").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
workspace=tempfile.TemporaryDirectory()
os.environ["DJANGO_SETTINGS_MODULE"]="gitlab_project.settings"
os.environ["DJANGO_DB_PATH"]=str(Path(workspace.name)/"git-records.sqlite3")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"]="true"
import django
django.setup()
from django.core.management import call_command
call_command("migrate",verbosity=0)
print("Isolated Git laboratory database ready")

## Original problem and interface
The aim remains branch creation, parallel commits, merging and conflict resolution. The original simulation and navigation remain the entry point. Accounts and records extend that same workflow.

In [ ]:
from django.test import Client
client=Client()
response=client.get("/")
assert response.status_code==200
assert b"Git Branching, Merging and Conflict Resolution" in response.content
assert b"SVU" not in response.content
assert client.get("/course/").status_code==302
print("Original Git experiment is the entry point")

## Validated Git practice records, ORM and CSV
Each record stores a branch/attempt name, quiz percentage and command/conflict notes. ModelForm validation runs before create or update; ownership limits each learner to their own records. CSV batches use the same validation before bulk_create.

In [ ]:
from course.models import User,Observation
from course.forms import ObservationForm
from course.services import import_csv
from django.core.files.uploadedfile import SimpleUploadedFile
learner=User.objects.create_user("git-student","git@example.test","PracticePassword!789")
assert not ObservationForm({"title":"main","value":101}).is_valid()
count=import_csv(SimpleUploadedFile("practice.csv",(ROOT/"docs/sample-observations.csv").read_bytes()),learner)
assert count==3
rows=list(Observation.objects.filter(owner=learner))
for row in rows: row.notes += " — reviewed"
Observation.objects.bulk_update(rows,["notes"])
print(list(Observation.objects.values_list("title","value")))

## Authentication, authorization and quiz persistence
The original quiz can save its result into the signed-in learner’s record. Instructor export requires a group permission. Admin supports filtering learners and batch actions.

In [ ]:
client.force_login(learner)
response=client.post("/course/attempts/save/",{"title":"Git quiz — main","value":75,"notes":"3/4 correct"})
assert response.status_code==201
assert client.get("/course/instructor/export/").status_code==403
from django.contrib.auth.models import Group,Permission
group,_=Group.objects.get_or_create(name="Instructors")
group.permissions.add(Permission.objects.get(codename="export_observation"))
learner.groups.add(group)
assert client.get("/course/instructor/export/").status_code==200
assert client.get("/course/csv/export/").status_code==200
pdf=client.get("/course/pdf/")
assert b"".join(pdf.streaming_content).startswith(b"%PDF")
print("Saved quiz, permissions, CSV and PDF verified")

## Templates, static assets and JSX
The Git experiment uses function views and templates; the merge prediction uses a class view and compiled local React JSX. The verbatim example preserves client-side delimiters. Static files are discoverable with findstatic and collectstatic.

In [ ]:
from django.contrib.staticfiles import finders
from django.template import Template,Context
assert finders.find("lab/app.js")
assert Template("{% verbatim %}{{ branch_name }}{% endverbatim %}").render(Context())=="{{ branch_name }}"
assert client.get("/course/react/").status_code==200
print("Git templates and static assets verified")

## Companion REST API
The independent loopback FastAPI service provides the same Git practice-record resource with typed body, path and query validation. It has a separate SQLite store and no shared Django authentication.

In [ ]:
from api import main as api
from fastapi.testclient import TestClient
api.DB_PATH=Path(workspace.name)/"api.sqlite3"
with TestClient(api.app) as api_client:
    created=api_client.post("/observations",json={"title":"feature/navbar","value":75,"notes":"Conflict resolved"})
    assert created.status_code==201
    assert api_client.get("/observations?min_value=80").json()==[]
    assert api_client.post("/observations",json={"title":"main","value":101}).status_code==422
    print(created.json())

## Verification and deployment
Run Django tests, FastAPI tests and static checks using README commands. Git server and deployment exercises are supporting procedures for this same project, not additional lab experiments. See docs/implementation-checklist.md for each requested implementation topic and its evidence.

In [ ]:
from django.db import connections
connections.close_all()
workspace.cleanup()
print("Temporary records removed")